# Country-Level Longer-Term Forecasting of First-Time Asylum Applications

A SARIMA/SARIMAX time-series exercise using Italy monthly Eurostat data

Source: Eurostat, first-time asylum applicants by citizenship, age and sex - monthly data: https://ec.europa.eu/eurostat/databrowser/view/migr_asyappctzm/default/table?lang=en

In [ ]:
from pathlib import Path
import subprocess

import matplotlib.pyplot as plt
import pandas as pd


DATA_RELATIVE_PATH = Path("data/processed/italy_first_time_asylum_monthly.csv")
REPOSITORY_URL = "https://github.com/kcsongor011/humanitarian_decision_toolkit.git"

try:
    import google.colab  # type: ignore  # noqa: F401

    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    PROJECT_ROOT = Path("/content/humanitarian_decision_toolkit")
    data_path = PROJECT_ROOT / DATA_RELATIVE_PATH
    if not data_path.exists():
        if not PROJECT_ROOT.exists():
            subprocess.run(
                ["git", "clone", REPOSITORY_URL, str(PROJECT_ROOT)],
                check=True,
            )
        if not data_path.exists():
            raise FileNotFoundError(f"Could not find the processed data at {data_path}.")
else:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / DATA_RELATIVE_PATH).exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError("Could not find the project root from the current working directory.")
    data_path = PROJECT_ROOT / DATA_RELATIVE_PATH

data_path

In [ ]:
applications = pd.read_csv(data_path, parse_dates=["date"])
applications.head()

In [ ]:
applications.tail()

In [ ]:
expected_months = pd.date_range(
    applications["date"].min(),
    applications["date"].max(),
    freq="MS",
)
frequency_complete = applications["date"].tolist() == expected_months.tolist()

pd.DataFrame(
    {
        "row_count": [len(applications)],
        "start_date": [applications["date"].min().date()],
        "end_date": [applications["date"].max().date()],
        "missing_applications": [int(applications["applications"].isna().sum())],
        "monthly_frequency_complete": [frequency_complete],
    }
)

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    applications["date"],
    applications["applications"],
    color="tab:blue",
    linewidth=1.2,
)
ax.set_title("Italy monthly first-time asylum applications")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(True, alpha=0.3)
plt.tight_layout()

In [ ]:
applications_with_rolling = applications.assign(
    rolling_12_month_average=applications["applications"].rolling(12).mean()
)

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(
    applications_with_rolling["date"],
    applications_with_rolling["applications"],
    color="tab:blue",
    linewidth=1,
    alpha=0.45,
    label="Monthly applications",
)
ax.plot(
    applications_with_rolling["date"],
    applications_with_rolling["rolling_12_month_average"],
    color="tab:orange",
    linewidth=2,
    label="12-month rolling average",
)
ax.set_title("Italy monthly applications with 12-month rolling average")
ax.set_xlabel("Month")
ax.set_ylabel("Applications")
ax.grid(True, alpha=0.3)
ax.legend()
plt.tight_layout()